# **Práctica 2**
## *Reconocimiento de patrones*


Autor:
 - Adrián Redondo García

---

# Introducción

En esta práctica, mediante un dataset vamos a entrenar dos modelos de reconocimiento de patrones siendo estos SVM y *k*-NN.

Antes de empezar con el entrenamiendo, deberemos limpiar y transformar los datos para que los modelos sean capaces de utilizarlos.

---

# Exploración, limpieza y transformación de datos

Primero habrá que hacer un procesor de limpieza y transformación para que los modelos puedan leer nuestros datos.

Importamos los archivos:

In [1]:
import pandas as pd
import numpy as np
import copy

df = pd.read_csv('ecommerce_dataset_10000.csv')

df

,customer_id,first_name,last_name,gender,age_group,signup_date,country,product_id,product_name,category,quantity,unit_price,order_id,order_date,order_status,payment_method,rating,review_text,review_id,review_date
0,CUST2353,Erica,Oliver,Female,Teenagers,2022-06-29,Canada,PROD108,Fitbit Versa 3,Electronics,3,229,ORD10000,2023-07-13,Pending,Credit Card,2,good,REV20000,2025-06-06
1,CUST4463,Christopher,White,Male,Adults,2023-08-24,China,PROD103,Levi's Jeans,Apparel,4,59,ORD10001,2024-08-12,Pending,PayPal,2,average,REV20001,2023-08-05
2,CUST4512,Spencer,Foster,Male,Senior,2023-07-18,Germany,PROD111,Lego Star Wars Set,Toys,2,59,ORD10002,2024-08-04,Delivered,Cash on Delivery,5,good,REV20002,2023-01-03
3,CUST5711,Jessica,Harris,Male,Teenagers,2025-08-22,France,PROD107,Dyson Vacuum,Home & Kitchen,4,399,ORD10003,2025-05-23,Delivered,Cash on Delivery,2,very good,REV20003,2023-03-14
4,CUST1296,Amy,Johnson,Female,Teenagers,2021-03-23,Brazil,PROD105,Adidas Running Shoes,Apparel,1,110,ORD10004,2023-07-02,Returned,Cash on Delivery,1,very good,REV20004,2023-10-18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,CUST2529,Lisa,Coleman,Male,Senior,2023-01-06,UK,PROD109,Kindle Paperwhite,Books,5,129,ORD19995,2023-06-07,Delivered,PayPal,4,very good,REV29995,2024-07-30
9996,CUST4602,Stacy,Brown,Other,Teenagers,2023-08-30,Canada,PROD109,Kindle Paperwhite,Books,5,129,ORD19996,2025-08-05,Returned,PayPal,1,bad,REV29996,2024-09-13
9997,CUST1448,Sheila,Roberts,Other,Adults,2024-05-18,USA,PROD107,Dyson Vacuum,Home & Kitchen,5,399,ORD19997,2023-12-23,Returned,Cash on Delivery,3,average,REV29997,2023-09-13
9998,CUST1343,Paul,Lam,Male,Teenagers,2024-07-14,Canada,PROD107,Dyson Vacuum,Home & Kitchen,3,399,ORD19998,2022-08-30,Cancelled,Credit Card,2,very good,REV29998,2025-07-07


Podemos ver la información específica del dataset:

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customer_id     10000 non-null  object
 1   first_name      10000 non-null  object
 2   last_name       10000 non-null  object
 3   gender          10000 non-null  object
 4   age_group       10000 non-null  object
 5   signup_date     10000 non-null  object
 6   country         10000 non-null  object
 7   product_id      10000 non-null  object
 8   product_name    10000 non-null  object
 9   category        10000 non-null  object
 10  quantity        10000 non-null  int64 
 11  unit_price      10000 non-null  int64 
 12  order_id        10000 non-null  object
 13  order_date      10000 non-null  object
 14  order_status    10000 non-null  object
 15  payment_method  10000 non-null  object
 16  rating          10000 non-null  int64 
 17  review_text     10000 non-null  object
 18  review_

Hacemos una copia del dataset para poder realizar modificaciones sin afectar al dataset original.

In [3]:
df1 = copy.deepcopy(df)

Ahora limpiaremos los datos, ya que algunas columnas no son relevantes a la hora de entrenar los modelos de predicción.

Quitaremos los identificadores únicos (como customer_id, order_id, etc), los nombres propios y texto plano (first_name, last_name y review_text) y las fechas.

In [4]:
cols_eliminadas = [
    "customer_id", "first_name", "last_name",
    "product_id", "order_id", "review_id",
    "review_text", "product_name",
    "signup_date", "order_date", "review_date"
]

df1 = df1.drop(columns=cols_eliminadas, errors="ignore")

# Resultado de la tabla con las columnas innecesarias eliminadas
df1.head()


,gender,age_group,country,category,quantity,unit_price,order_status,payment_method,rating
0,Female,Teenagers,Canada,Electronics,3,229,Pending,Credit Card,2
1,Male,Adults,China,Apparel,4,59,Pending,PayPal,2
2,Male,Senior,Germany,Toys,2,59,Delivered,Cash on Delivery,5
3,Male,Teenagers,France,Home & Kitchen,4,399,Delivered,Cash on Delivery,2
4,Female,Teenagers,Brazil,Apparel,1,110,Returned,Cash on Delivery,1


Tras todo esto, podemos elegir una columna objetivo.

En mi caso he decidido la columna "category" ya que se podría predecir el tipo de producto basándonos en los datos del cliente como método de pago, país, género, grupo de edad, etc.

In [5]:
objetivo = "category"

# Características que los modelos usarán para predecir
X = df1.drop(columns=[objetivo])

# Objetivo a predecir
y = df1[objetivo]

Para que los modelos de scikit-learn puedan trabajar con los datos, dividimos el dataset en dos grupos:
 - Numéricos (en mi caso en el dataset hay int64 y float64)
 - Categóricos (en caso en el dataset solamente hay object)

Con esto los modelos podrán ser entrenados sin problemas.

In [6]:
variables_numericas = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
variables_categoricas = X    .select_dtypes(include=["object"]).columns.tolist()

Tras separar el dataset en dos grupos, preparamos los datos para los modelos.

Para los datos numéricos si falta algun dato en la columna lo sustituimos por la mediana y tras eso los normalizamos.
Para los datos categóricos si falta algun dato lo sustituimos por el más frecuente y tras eso convierte el texto a columnas binarias.

Lo siguente es combinar ambos preprocesadores para que así para cada tipo de columna se aplice su preprocesamiento correspondiente.

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Numérico
trans_numerico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categórico
trans_categorico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combinación en un único preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ("num", trans_numerico, variables_numericas),
        ("cat", trans_categorico, variables_categoricas)
    ]
)

print("Preprocesador creado")


Preprocesador creado


Ahora separaremos los datos en dos partes, una parte será para entrenar a los modelos y la otra para comprobar que tan precisos son.

In [8]:
from sklearn.model_selection import train_test_split

entrenamiento_X, test_X, entrenamiento_y, test_y = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=5,
    stratify=y
)

Con todo ya preparado podemos pasar a los modelos.

---

# SVM y *k*-NN

## SVN

SVN intenta separar las clases de datos maximizando la distancia entre la frontera y los datos de cada clase, ya sea una línea, plano o hiperplano.

Para ello se tiene en cuenta:
 - Margen: espacio entre la frontera y las muestras más cercanas. Si el margen es grande el modelo es más robusto y si es pequeño significa que hay sobreajuste.
 - Vectores de soporte: los 
 - C: controla que tan estricto es el modelo. Si es bajo permite más errores y si es grande es más estricto.

La frontera puede ser de dos tipos:
 - Lineal
 - Curva

In [9]:
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

Creamos un pipeline el cual une el preprocesador y el clasificador SVC.

In [10]:
pipeline_svc = make_pipeline(preprocessor, SVC())

Tras eso, definimos una regilla de hiperparámetros para que GridSearchCV pruebe todas las combinaciones y seleccione la que mejor se adapta a nuestros datos.

In [11]:
parametros_svc = {
    "svc__C": [0.1, 1, 10],             # Rigidez
    "svc__kernel": ["linear", "rbf"],   # Frontera de decisión
    "svc__gamma": ["scale", "auto"]     # Curvatura
}


Por último entrenamos el modelo.

In [12]:
grid_svc = GridSearchCV(
    estimator=pipeline_svc,
    param_grid=parametros_svc,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=3
)

# Entrenamiento
grid_svc.fit(entrenamiento_X, entrenamiento_y)

print("\nFin del entrenamiento")

Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV 1/5] END svc__C=0.1, svc__gamma=scale, svc__kernel=linear;, score=0.466 total time=   2.4s
[CV 4/5] END svc__C=0.1, svc__gamma=scale, svc__kernel=linear;, score=0.466 total time=   2.4s
[CV 3/5] END svc__C=0.1, svc__gamma=scale, svc__kernel=linear;, score=0.466 total time=   2.4s
[CV 2/5] END svc__C=0.1, svc__gamma=scale, svc__kernel=linear;, score=0.466 total time=   2.4s
[CV 1/5] END svc__C=0.1, svc__gamma=auto, svc__kernel=linear;, score=0.466 total time=   2.4s
[CV 3/5] END svc__C=0.1, svc__gamma=auto, svc__kernel=linear;, score=0.466 total time=   2.4s
[CV 4/5] END svc__C=0.1, svc__gamma=auto, svc__kernel=linear;, score=0.466 total time=   2.4s
[CV 2/5] END svc__C=0.1, svc__gamma=auto, svc__kernel=linear;, score=0.466 total time=   2.5s
[CV 5/5] END svc__C=0.1, svc__gamma=scale, svc__kernel=linear;, score=0.466 total time=   2.7s
[CV 5/5] END svc__C=0.1, svc__gamma=auto, svc__kernel=linear;, score=0.466 total time=  

Podemos ver los resultados en la siguiente celda:

In [13]:
mejor_svm = grid_svc.best_estimator_
prediccion_svm = mejor_svm.predict(test_X)

print("Resultados:")
print(f"Exactitud: {accuracy_score(test_y, prediccion_svm)}")

print(f"\nReporte:\n {classification_report(test_y, prediccion_svm, zero_division=0)}")

print(f"\nMatriz de confusión:\n {confusion_matrix(test_y, prediccion_svm)}")


Resultados:
Exactitud: 0.524

Reporte:
                 precision    recall  f1-score   support

       Apparel       0.36      0.91      0.52       614
         Books       0.24      0.04      0.07       400
   Electronics       0.76      0.99      0.86       785
Home & Kitchen       1.00      0.39      0.56       417
        Sports       0.45      0.11      0.18       390
          Toys       0.12      0.03      0.05       394

      accuracy                           0.52      3000
     macro avg       0.49      0.41      0.37      3000
  weighted avg       0.52      0.52      0.45      3000


Matriz de confusión:
 [[561  24   0   0  20   9]
 [362  17   0   0   7  14]
 [  0   0 774   0   0  11]
 [185   8  58 164   1   1]
 [273  18   0   0  44  55]
 [163   3 190   0  26  12]]


Se puede ver que el modelo saca una exactitud del 52%, luego hacierta la mitad de las veces.

Analizando por categorías, Electronics tiene un recall muy alto mientras que otras como Books o Toys tienen uno muy bajo, luego para estas el modelo no tiene suficiente información para diferenciarlas.

En la matriz de confusión se puede ver que se muchas categorías se confunden entre sí.

Con los datos que tenemos, category es una columna dificil de predecir. Un motivo podría ser debido a que no aparecen con la misma frecuencia todas las categorías, por ejemplo, aparece con más frecuencia Electronics que Toys.

## *k*-NN

*k*-NN (k-Nearest Neighbors) clasifica cada nueva muestra comparándola con el conjunto de entrenamiento dado.

Calcula la distancia entre el punto nuevo y todos los datos y mediante los *k* más cercanos coloca la clase más frecuente.

Se tienen en cuenta:
 - Nº de vecinos: el número de vecinos más cercanos que tiene en cuenta el modelo
 - Peso: como influyen los vecinos a la hora de tomar decisiones
 - Distancia: cómo se mide la distancia entre los vecinos

Creamos un pipeline el cual une el preprocesador y el clasificador *k*-NN.

In [14]:
pipeline_knn = make_pipeline(preprocessor, KNeighborsClassifier())

Tras eso, definimos una regilla de hiperparámetros para que GridSearchCV pruebe todas las combinaciones y seleccione la que mejor se adapta a nuestros datos.

In [15]:
parametros_knn = {
    "kneighborsclassifier__n_neighbors": [3, 5, 7, 9, 11, 13, 15],  # nº de vecinos
    "kneighborsclassifier__weights": ["uniform", "distance"],       # como afectan los vecinos
    "kneighborsclassifier__metric": ["euclidean", "manhattan"]      # como se mide la distancia entre los vecinos
}


Por último, entrenamos al modelo:

In [16]:
grid_knn = GridSearchCV(
    estimator=pipeline_knn,
    param_grid=parametros_knn,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=3
)

# Entrenamiento
grid_knn.fit(entrenamiento_X, entrenamiento_y)

print("\nFin del entrenamiento")

Fitting 5 folds for each of 28 candidates, totalling 140 fits
[CV 1/5] END kneighborsclassifier__metric=euclidean, kneighborsclassifier__n_neighbors=3, kneighborsclassifier__weights=distance;, score=0.341 total time=   0.9s
[CV 3/5] END kneighborsclassifier__metric=euclidean, kneighborsclassifier__n_neighbors=3, kneighborsclassifier__weights=uniform;, score=0.377 total time=   0.9s
[CV 4/5] END kneighborsclassifier__metric=euclidean, kneighborsclassifier__n_neighbors=3, kneighborsclassifier__weights=uniform;, score=0.358 total time=   0.9s
[CV 5/5] END kneighborsclassifier__metric=euclidean, kneighborsclassifier__n_neighbors=3, kneighborsclassifier__weights=distance;, score=0.346 total time=   0.9s
[CV 1/5] END kneighborsclassifier__metric=euclidean, kneighborsclassifier__n_neighbors=3, kneighborsclassifier__weights=uniform;, score=0.338 total time=   0.9s
[CV 1/5] END kneighborsclassifier__metric=euclidean, kneighborsclassifier__n_neighbors=5, kneighborsclassifier__weights=distance;, 

Podemos ver los resultados en la siguiente celda:

In [17]:
mejor_knn = grid_knn.best_estimator_
prediccion_knn = mejor_knn.predict(test_X)

print("Resultados:")
print(f"Exactitud: {accuracy_score(test_y, prediccion_knn)}")

print(f"\nReporte:\n{classification_report(test_y, prediccion_knn, zero_division=0)}")

print(f"\nMatriz de confusión:\n{confusion_matrix(test_y, prediccion_knn)}")


Resultados:
Exactitud: 0.3923333333333333

Reporte:
                precision    recall  f1-score   support

       Apparel       0.30      0.65      0.41       614
         Books       0.21      0.19      0.20       400
   Electronics       0.71      0.66      0.69       785
Home & Kitchen       0.55      0.24      0.34       417
        Sports       0.25      0.12      0.16       390
          Toys       0.17      0.08      0.11       394

      accuracy                           0.39      3000
     macro avg       0.37      0.33      0.32      3000
  weighted avg       0.41      0.39      0.37      3000


Matriz de confusión:
[[398  74  32  20  43  47]
 [235  77  16  13  32  27]
 [152  43 521  16  23  30]
 [174  39  66 101  17  20]
 [199  67  32  17  48  27]
 [180  66  69  15  32  32]]


Vemos que el modelo tiene una exactitud del 39%, menor a la obtenida con SVM.

Analizando por categorías, Electronics tiene un recall muy alto mientras que otras como Books o Toys tienen uno muy bajo, luego para estas el modelo no tiene suficiente información para diferenciarlas.

Podemos sacar las mismas conclusiones que con SVM.

# Conclusiones

Hemos entrenado dos modelos de reconocimiento de patrones, SVM el cual separa las clases maximizando la distancia entre la frontera y los datos más cercanos y *k*-NN el cual se enfoca en comparar mediante distancias nuevos puntos con el conjunto de entrenamiento.

Vemos que los modelos no predicen del todo bien la columna objetivo a pesar de que con la información sobre el país de origen, método de pago, grupo de edad... podríamos ser capaces de predecir la categoría, pero se ha visto que no.

Esto puede ser a que no tenemos suficientes datos en el dataset, los datos no están balanceados (ya que se puede ver que abundan mucho los de tipo Electronics, Apparel... y no tanto Books, Toys...) o que no hay mucha correlación entre los datos del dataset como se había pensado al principio, luego la predicción no puede llegar a ser muy exacta.